# 🎯 Netelpro + RAFT: Entrenando Qwen2.5-1.5B a escribir programas Netelpro correctos
### *Rejection-sampling fine-tuning (RAFT/STaR): la recompensa es "compiló + pasó los tests", no preferencia humana*

Ver `docs/superpowers/specs/2026-09-07-rlvr-netelpro-raft-design.md` para el diseño completo.

**Loop por ronda:** muestrear N candidatos por tarea de train -> verificar (compiló + pasó todos los casos) -> quedarse con los que pasan -> SFT LoRA sobre esos -> repetir con el modelo mejorado. 5 rondas con pool acumulado (v2, RAFT canónico), medido contra un split held-out (OOD) que el loop nunca entrena.

---
### ⚙️ Requisitos previos en Google Colab:
1. `Entorno de ejecución` -> `Cambiar tipo de entorno de ejecución` -> **T4 GPU** (gratis).
2. Ejecutar cada celda en orden con `Shift + Enter`.

## 1. Instalación de dependencias

In [ ]:
# Unsloth + TRL para fine-tune LoRA rápido en T4, llvmlite para netelpro
!pip install --no-deps "xformers<0.0.29" "trl<0.15.0" peft accelerate bitsandbytes triton
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q llvmlite>=0.49


### 1b. Verificar GPU (antes de importar unsloth)

In [ ]:
# Unsloth se niega a importar sin acelerador CUDA, con un NotImplementedError criptico.
# Fallamos temprano y en claro: sin GPU no hay experimento.
import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        "No hay GPU activa. En Colab: Entorno de ejecucion -> Cambiar tipo de entorno "
        "-> T4 GPU -> Guardar (la VM se reinicia: corre todo de nuevo desde la celda 1)."
    )
print("GPU OK:", torch.cuda.get_device_name(0))


## 2. Clonar el repo y cargar el corpus de tareas RLVR

In [ ]:
# Clonamos el repo para tener netelpro/ y rlvr/ disponibles como paquetes
!git clone https://github.com/jona2428/netelpro.git

import sys
sys.path.insert(0, "netelpro")

from rlvr.tasks import load_all_tasks, split_train_ood
from rlvr.prompting import build_prompt
from rlvr.verify import verify_program

all_tasks = load_all_tasks()
train_ids, ood_ids = split_train_ood(list(all_tasks.keys()), ood_fraction=0.2)
from rlvr.tasks import OOD_TASK_IDS
assert sorted(ood_ids) == sorted(OOD_TASK_IDS), "held-out explicito: split debe calzar con OOD_TASK_IDS"
print(f"Corpus: {len(all_tasks)} tareas -- {len(train_ids)} train, {len(ood_ids)} OOD (held-out)")


## 3. Configuración del experimento

In [ ]:
NUM_ROUNDS = 5  # v2: 3->5 -- el plateau de la v1 en ronda 2 sugiere falta de señal
SAMPLES_PER_TASK = 16  # v2: 8->16 -- más candidatos por tarea, más señal de SFT por ronda
MAX_KEEP_PER_TASK = 2  # tope de candidatos que pasan por tarea, evita desbalancear el SFT
MAX_NEW_TOKENS = 256
SAMPLING_TEMPERATURE = 0.8
NUM_TEST_CASES = 20
EVAL_SEED = 0
MAX_VERIFY_STEPS = 1_000_000  # presupuesto de pasos del verificador (fix C1: runaway tail-recursion no cuelga Colab)
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"  # base limpio, NO el checkpoint DPO de honestidad (spec Sec 1, Sec 9)


## 4. Cargar el modelo base + LoRA
Mismo patrón que `train_colab.ipynb` (Unsloth 4-bit + LoRA r=16).

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)


## 5. Muestreo y extracción de código Netelpro

In [ ]:
def extract_sl_code(raw_text: str) -> str:
    """El modelo puede envolver el programa en un bloque de código markdown --
    si hay un fence ``` lo extraemos, si no devolvemos el texto tal cual."""
    if "```" in raw_text:
        parts = raw_text.split("```")
        if len(parts) >= 2:
            candidate = parts[1]
            candidate = candidate.removeprefix("netelpro").removeprefix("lisp").strip()
            return candidate
    return raw_text.strip()


def sample_completions(prompt: str, n: int, temperature: float) -> list[str]:
    FastLanguageModel.for_inference(model)
    chat = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer([chat] * n, return_tensors="pt", padding=True).to("cuda")
    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=True,
        temperature=temperature,
    )
    texts = tokenizer.batch_decode(
        outputs[:, inputs.input_ids.shape[1] :], skip_special_tokens=True
    )
    return [extract_sl_code(t) for t in texts]


## 6. Pass rate en el split OOD (criterio de éxito, spec §7)

In [ ]:
def evaluate_pass_rate(task_ids: list[str], num_samples: int) -> tuple[float, list[str]]:
    # task_id ES la clave de all_tasks (load_all_tasks() carga por
    # nombre de módulo, y por construcción TASK_ID == nombre de módulo
    # en todo el corpus -- ver rlvr/tasks/*.py). Sin indirección.
    # Seed fija antes de muestrear (v2): baseline, rondas y eval final comparten
    # los draws del sampler -> comparación pareada (caveat de la v1 cerrado).
    torch.manual_seed(EVAL_SEED)
    passed_ids: list[str] = []
    for task_id in task_ids:
        task_module = all_tasks[task_id]
        prompt = build_prompt(task_module)
        candidates = sample_completions(prompt, num_samples, SAMPLING_TEMPERATURE)
        if any(
            verify_program(
                c, task_module, num_cases=NUM_TEST_CASES, seed=EVAL_SEED, max_steps=MAX_VERIFY_STEPS
            ).passed
            for c in candidates
        ):
            passed_ids.append(task_id)
    # v3: devuelve también QUÉ tareas pasaron, no solo el %, para el reporte
    # con detalle por tarea (celda 10) -- antes esa info se descartaba.
    return len(passed_ids) / len(task_ids), passed_ids


baseline_pass_rate, baseline_passed_ids = evaluate_pass_rate(ood_ids, num_samples=SAMPLES_PER_TASK)
print(f"[baseline, sin entrenar] pass rate OOD: {baseline_pass_rate:.1%}")
print(f"[baseline] tareas resueltas: {baseline_passed_ids}")


## 7. Loop RAFT iterativo**v2 -- pool acumulado:** cada ronda SFT entrena sobre TODO el harvest verificado hasta ahora (no solo el fresco) -- RAFT canónico; evita el forgetting que planchó la ronda 2 de la v1.

In [ ]:
from trl import SFTConfig, SFTTrainer
from datasets import Dataset

all_sft_examples: list[dict] = []  # v2: pool acumulado -- RAFT canónico
round_history: list[dict] = []  # v3: trayectoria completa para el reporte (celda 10)
for round_num in range(NUM_ROUNDS):
    print(f"\n=== Ronda {round_num} ===")
    sft_examples = []
    for task_id in train_ids:
        task_module = all_tasks[task_id]
        prompt = build_prompt(task_module)
        candidates = sample_completions(prompt, SAMPLES_PER_TASK, SAMPLING_TEMPERATURE)
        kept = 0
        for candidate in candidates:
            if kept >= MAX_KEEP_PER_TASK:
                break
            result = verify_program(candidate, task_module, num_cases=NUM_TEST_CASES, seed=EVAL_SEED, max_steps=MAX_VERIFY_STEPS)
            if result.passed:
                sft_examples.append({"prompt": prompt, "completion": candidate})
                kept += 1
    all_sft_examples.extend(sft_examples)  # v2: acumular, nunca descartar lo verificado
    print(f"Ronda {round_num}: {len(sft_examples)} nuevos -- pool acumulado: {len(all_sft_examples)}")

    if not sft_examples:
        print("Ninguna tarea de train produjo un candidato que pasara -- se aborta esta ronda")
        continue

    round_dataset = Dataset.from_list(all_sft_examples)  # v2: entrena sobre el pool completo
    FastLanguageModel.for_training(model)

    def format_sft_example(example):
        # Combina prompt + completion en UNA sola secuencia de entrenamiento,
        # usando el mismo apply_chat_template que sample_completions (celda 5) --
        # si el formato de entrenamiento diverge del de muestreo, el modelo
        # entrena en un formato y samplea en otro.
        chat = tokenizer.apply_chat_template(
            [{"role": "user", "content": example["prompt"]}],
            tokenize=False,
            add_generation_prompt=True,
        )
        return chat + example["completion"]

    sft_args = SFTConfig(
        output_dir=f"netelpro_raft_round_{round_num}",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=2,
        logging_steps=1,  # v2: rondas cortas -- con 5 la tabla de loss salía vacía
        save_strategy="no",
        warmup_ratio=0.1,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        report_to="none",
        # Dataset rows carry prompt/completion keys, so TRL classifies it as
        # prompt-completion and defaults completion_only_loss=True -- which
        # Unsloth's fork rejects alongside formatting_func. Explicit False
        # selects full-sequence loss (prompt + completion), the canonical
        # RAFT objective, and unblocks the formatter path.
        completion_only_loss=False,
    )
    sft_trainer = SFTTrainer(
        model=model,
        args=sft_args,
        train_dataset=round_dataset,
        formatting_func=format_sft_example,
    )
    sft_trainer.train()

    round_pass_rate, round_passed_ids = evaluate_pass_rate(ood_ids, num_samples=SAMPLES_PER_TASK)
    round_history.append({
        "round": round_num,
        "pass_rate": round_pass_rate,
        "passed_ids": round_passed_ids,
        "pool_size": len(all_sft_examples),
    })
    print(f"[ronda {round_num}] pass rate OOD: {round_pass_rate:.1%} -- resueltas: {round_passed_ids}")


## 8. Criterio de éxito (spec §7)
El RAFT tiene que superar CLARAMENTE al baseline sin entrenar -- si no, no está enseñando nada que el prompt no diera gratis.

In [ ]:
final_pass_rate, final_passed_ids = evaluate_pass_rate(ood_ids, num_samples=SAMPLES_PER_TASK)
print(f"Baseline (sin entrenar): {baseline_pass_rate:.1%}")
print(f"Final (tras {NUM_ROUNDS} rondas de RAFT): {final_pass_rate:.1%}")
if final_pass_rate > baseline_pass_rate + 0.1:
    print("RAFT superó claramente al baseline -- la señal de recompensa enseñó algo real.")
else:
    print("RAFT NO superó claramente al baseline -- no declarar éxito, reportar el número tal cual.")


## 9. Exportar a GGUF

In [ ]:
model.save_pretrained_gguf(
    "netelpro_qwen1.5b_raft", tokenizer, quantization_method="q4_k_m"
)
print("✅ Modelo GGUF exportado en la carpeta 'netelpro_qwen1.5b_raft'.")


## 10. Reporte final (trayectoria + detalle por tarea)

In [ ]:
import datetime

report_lines = []
now = datetime.datetime.now(datetime.timezone.utc).isoformat()
report_lines.append(f"# Netelpro RAFT -- reporte de corrida ({now})")
report_lines.append("")
report_lines.append(f"- Corpus: {len(all_tasks)} tareas -- {len(train_ids)} train, {len(ood_ids)} OOD")
report_lines.append(
    f"- Rondas: {NUM_ROUNDS}, muestras/tarea: {SAMPLES_PER_TASK}, "
    f"pool SFT final: {len(all_sft_examples)} ejemplos"
)
report_lines.append("")
report_lines.append(f"## Trayectoria OOD (pass@{SAMPLES_PER_TASK}, seed={EVAL_SEED})")
report_lines.append("")
report_lines.append("| Etapa | Pass rate | Tareas resueltas |")
report_lines.append("|---|---|---|")
report_lines.append(
    f"| Baseline | {baseline_pass_rate:.1%} | {len(baseline_passed_ids)}/{len(ood_ids)} |"
)
for entry in round_history:
    report_lines.append(
        f"| Ronda {entry['round']} | {entry['pass_rate']:.1%} | "
        f"{len(entry['passed_ids'])}/{len(ood_ids)} |"
    )
report_lines.append(
    f"| Final | {final_pass_rate:.1%} | {len(final_passed_ids)}/{len(ood_ids)} |"
)
report_lines.append("")
report_lines.append("## Detalle por tarea OOD (baseline vs final)")
report_lines.append("")
report_lines.append("| Tarea | Baseline | Final |")
report_lines.append("|---|---|---|")
for tid in sorted(ood_ids):
    b = "si" if tid in baseline_passed_ids else "no"
    f = "si" if tid in final_passed_ids else "no"
    marker = " <-- curriculum gcd (run #3)" if tid == "gcd_pair" else ""
    report_lines.append(f"| {tid}{marker} | {b} | {f} |")
report_lines.append("")
if final_pass_rate > baseline_pass_rate + 0.1:
    report_lines.append("**Veredicto:** RAFT superó claramente al baseline.")
else:
    report_lines.append(
        "**Veredicto:** RAFT NO superó claramente al baseline -- reportado tal cual, "
        "sin declarar éxito."
    )

report_text = "\n".join(report_lines)
with open("informe_run.md", "w", encoding="utf-8") as f:
    f.write(report_text)

print(report_text)


## 11. Descargar resultados (GGUF + reporte)

In [ ]:
# Empaqueta el GGUF + el reporte y los baja directo al navegador -- sobrevive
# aunque la VM se desconecte apenas termine (Colab free tier corta sesiones
# inactivas; bajar a mano desde el explorador de archivos es un paso que se
# puede perder si eso pasa antes de que alguien lo haga).
import shutil
from google.colab import files

shutil.make_archive("netelpro_qwen1.5b_raft", "zip", "netelpro_qwen1.5b_raft")
files.download("netelpro_qwen1.5b_raft.zip")
files.download("informe_run.md")
